# 03 — Income, rent and operations

**Pregunta humana:** ¿La operación inmobiliaria se sostiene por sí misma?

Este reporte aísla la performance operativa: rentas, OPEX de propiedad y resultado operativo.  
No incluye funding, deuda, repagos, dividendos ni gasto personal.

Este notebook es el lugar correcto para discutir:

- renta total y por propiedad;
- impuestos, servicios, mantenimiento y legal;
- margen operativo;
- OPEX / renta;
- 2024 como año de presión operativa;
- 2026 como YTD si corresponde;
- QA de clasificación semántica.


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 260)

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "Makefile").exists() and (p / "accounting").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not find repo root. Run from inside accounting-backend.")

reports_dir = repo_root / "accounting" / "notebooks" / "accounting_reports"
if str(reports_dir) not in sys.path:
    sys.path.insert(0, str(reports_dir))

from _shared import *

repo_root = find_repo_root(repo_root)
pack_dir = professional_pack_dir(repo_root)

artifact_inventory = inspect_artifacts(repo_root)
metrics = load_annual_dashboard_metrics(repo_root)
annual_qa = load_annual_dashboard_qa(repo_root)
debt_status = load_debt_status_reconciliation(repo_root)

years = available_years(metrics)
currencies = available_currencies(metrics)
extended_qa = extended_qa_findings(metrics, artifact_inventory, debt_status)

print("repo_root:", repo_root)
print("pack_dir:", pack_dir)
print("years:", years)
print("currencies:", currencies)
print("metric rows:", len(metrics))


repo_root: /home/matias/repos/accounting-backend
pack_dir: /home/matias/repos/accounting-backend/out/professional_pack/latest
years: ['2022', '2023', '2024', '2025', '2026']
currencies: ['ARS', 'N/A', 'USD']
metric rows: 287


## 1. QA operativo

El foco acá es clasificación, unknown/review-required, posibles leaks de OPEX y métricas disponibles sin valor.


In [2]:
operating_qa = extended_qa[
    extended_qa["area"].isin(["metrics", "currency", "display"])
].copy()
display(short_status_table(operating_qa))
export_table(operating_qa, repo_root, "income_operations_extended_qa.csv")

dq_ops = metrics[
    metrics["metric_id"].astype(str).isin([
        "DQ.CLASSIFICATION.COVERAGE",
        "DQ.UNKNOWN.AMOUNT",
        "DQ.OPEX.LEAKAGE.AMOUNT",
    ])
].copy()
display(dq_ops.head(100))


,severity,area,check,n,detail
2,ok,currency,no_cross_currency_totals,0,No suspicious cross-currency Currency labels f...
1,ok,metrics,available_metric_has_value,0,No available metrics with NaN values
4,warning,display,hidden_metric_id_collisions,50,50 display key groups map to multiple metric_i...
5,warning,metrics,unavailable_visible,3,3 unavailable/blocked rows should be surfaced ...


,metric_id,period_grain,period,period_start,period_end,Currency,value,value_status,flow_or_stock,accounting_section,dashboard_section,dimension_name,dimension_value,source_table,source_filter,calculation_rule,frontend_suitability,public_flag,internal_flag,legacy_flag,validation_status,caveat,run_id,as_of_date,format_hint,metric_nature,aggregation_policy
63,DQ.UNKNOWN.AMOUNT,Y,2022,2022-01-01,2022-12-31,ARS,0.000000e+00,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
64,DQ.UNKNOWN.AMOUNT,Y,2023,2023-01-01,2023-12-31,ARS,0.000000e+00,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
65,DQ.UNKNOWN.AMOUNT,Y,2023,2023-01-01,2023-12-31,USD,0.000000e+00,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
66,DQ.UNKNOWN.AMOUNT,Y,2024,2024-01-01,2024-12-31,ARS,2.164500e+06,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
67,DQ.UNKNOWN.AMOUNT,Y,2024,2024-01-01,2024-12-31,USD,1.930000e+03,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
68,DQ.UNKNOWN.AMOUNT,Y,2025,2025-01-01,2025-12-31,ARS,0.000000e+00,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
69,DQ.UNKNOWN.AMOUNT,Y,2025,2025-01-01,2025-12-31,USD,0.000000e+00,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
70,DQ.UNKNOWN.AMOUNT,Y,2026,2026-01-01,2026-12-31,ARS,4.363840e+05,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
71,DQ.UNKNOWN.AMOUNT,Y,2026,2026-01-01,2026-12-31,USD,2.400000e+02,available,quality,coverage,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=unknown_or_ambiguous_outflows,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,
81,DQ.CLASSIFICATION.COVERAGE,Y,2022,2022-01-01,2022-12-31,ARS,1.000000e+00,available,quality,data_quality,6. Data quality and caveats,,,monthly_operating_statement.csv,statement_line=classification_coverage,last valid monthly coverage value in year,safe_with_caveat,True,False,False,ok,,LIVE_20260701T191005Z,2026-07-01,,,


## 2. Estado de resultado operativo

Estructura contable/gerencial:

```text
Renta total
- OPEX propiedad
= Resultado operativo neto
```

Lectura profesional: este cuadro responde si el patrimonio se sostiene operativamente. No responde si la familia está drenando, fondeando o endeudándose.


In [3]:
operating_specs = [
    spec("Income", "Ingresos operativos", "Renta total", "IS.RENT.TOTAL", 100,
         professional_comment="Ingreso operativo propio de propiedades. Excluye funding y deuda."),
    spec("Income", "Ingresos operativos", "Renta por propiedad", "IS.RENT.BY_PROPERTY", 120,
         append_dimension=True,
         professional_comment="Permite ver qué propiedades explican la renta. Conviene agregar % del total en una iteración posterior."),
    spec("Income", "Costos operativos", "OPEX propiedad", "IS.OPEX.PROPERTY", 200,
         professional_comment="Subtotal de costos reales de propiedad."),
    spec("Income", "Costos operativos", "OPEX por categoría", "IS.OPEX.BY_CATEGORY", 220,
         append_dimension=True,
         professional_comment="Descomposición por impuestos, servicios, mantenimiento, legal y otros."),
    spec("Income", "Resultado", "Resultado operativo neto", "IS.NET.OPERATING", 300,
         professional_comment="Renta menos OPEX. Medida principal de performance operativa."),
]

operating_table = build_statement_table(metrics, operating_specs, years, include_debug_cols=False, drop_all_empty_years=False)

operating_table = add_ratio_row(
    operating_table,
    section="Resultado",
    line="Margen operativo",
    numerator_line_contains="Resultado operativo neto",
    denominator_line_contains="Renta total",
    years=years,
    currency="ARS",
    professional_comment="Resultado operativo neto / renta total. Ayuda a separar crecimiento nominal de eficiencia operativa."
)
operating_table = add_ratio_row(
    operating_table,
    section="Resultado",
    line="OPEX / renta",
    numerator_line_contains="OPEX propiedad",
    denominator_line_contains="Renta total",
    years=years,
    currency="ARS",
    professional_comment="OPEX propiedad / renta total. Identifica años de presión operativa como 2024."
)

displayed_operating = display_statement(
    operating_table,
    "Estado de resultado operativo",
    "Ingresos y costos propios de la operación patrimonial. No incluye funding, retiros ni deuda.",
)
export_table(operating_table, repo_root, "income_operating_statement.csv")


## Estado de resultado operativo

Ingresos y costos propios de la operación patrimonial. No incluye funding, retiros ni deuda.

section,line,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat
Costos operativos,OPEX propiedad,ARS,587.266,2.392.541,8.066.302,11.955.361,6.739.472,available,Subtotal de costos reales de propiedad.,
Costos operativos,OPEX propiedad,USD,s/d,0,0,0,0,available,Subtotal de costos reales de propiedad.,
Costos operativos,OPEX por categoría — semantic_subbucket: Impuestos,ARS,227.940,1.541.706,4.183.410,6.876.980,5.414.569,available,"Descomposición por impuestos, servicios, mantenimiento, legal y otros.",
Costos operativos,OPEX por categoría — semantic_subbucket: Legal,ARS,37.700,277.093,1.044.210,147.000,s/d,available,"Descomposición por impuestos, servicios, mantenimiento, legal y otros.",
Costos operativos,OPEX por categoría — semantic_subbucket: Mantenimiento,ARS,215.900,57.500,592.174,1.131.802,s/d,available,"Descomposición por impuestos, servicios, mantenimiento, legal y otros.",
Costos operativos,OPEX por categoría — semantic_subbucket: Servicios,ARS,105.726,516.242,2.246.508,3.799.578,1.324.903,available,"Descomposición por impuestos, servicios, mantenimiento, legal y otros.",
Ingresos operativos,Renta total,ARS,4.948.804,9.141.302,13.558.336,40.125.422,29.589.000,available,Ingreso operativo propio de propiedades. Excluye funding y deuda.,
Ingresos operativos,Renta total,USD,s/d,s/d,3.610,4.560,2.280,available,Ingreso operativo propio de propiedades. Excluye funding y deuda.,
Ingresos operativos,Renta por propiedad — Lugar: CABA,ARS,982.000,1.710.000,s/d,800.000,1.700.000,available,Permite ver qué propiedades explican la renta. Conviene agregar % del total en una iteración posterior.,
Ingresos operativos,Renta por propiedad — Lugar: CABA,USD,s/d,s/d,3.610,4.560,2.280,available,Permite ver qué propiedades explican la renta. Conviene agregar % del total en una iteración posterior.,


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/income_operating_statement.csv')

## 3. Drilldown: renta por propiedad

Este cuadro permite ver concentración de renta por lugar/unidad.  
La mejora posterior recomendada es agregar `% del total anual` por propiedad, sin mezclar monedas.


In [4]:
rent_by_property = build_statement_table(
    metrics,
    [spec("Income", "Renta por propiedad", "Renta por propiedad", "IS.RENT.BY_PROPERTY", 100,
          append_dimension=True,
          professional_comment="Renta anual por propiedad/unidad según dimensión disponible.")],
    years,
    include_debug_cols=True,
    drop_all_empty_years=True,
)
display_statement(rent_by_property, "Renta por propiedad", "Detalle dimensionado. Debug cols visibles para validar etiquetas y fuentes.", debug=True)
export_table(rent_by_property, repo_root, "income_rent_by_property.csv")


## Renta por propiedad

Detalle dimensionado. Debug cols visibles para validar etiquetas y fuentes.

section,line,metric_id,dimension_name,dimension_value,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat,source_table,format_hint
Renta por propiedad,Renta por propiedad — Lugar: CABA,IS.RENT.BY_PROPERTY,Lugar,CABA,ARS,982.000,1.710.000,s/d,800.000,1.700.000,available,Renta anual por propiedad/unidad según dimensión disponible.,,monthly_flow_semantic_split.csv,number
Renta por propiedad,Renta por propiedad — Lugar: CABA,IS.RENT.BY_PROPERTY,Lugar,CABA,USD,s/d,s/d,3.610,4.560,2.280,available,Renta anual por propiedad/unidad según dimensión disponible.,,monthly_flow_semantic_split.csv,number
Renta por propiedad,Renta por propiedad — Lugar: Tigre 01,IS.RENT.BY_PROPERTY,Lugar,Tigre 01,ARS,1.686.700,3.595.931,5.975.753,18.021.572,12.899.000,available,Renta anual por propiedad/unidad según dimensión disponible.,,monthly_flow_semantic_split.csv,number
Renta por propiedad,Renta por propiedad — Lugar: Tigre 28,IS.RENT.BY_PROPERTY,Lugar,Tigre 28,ARS,1.223.120,2.233.800,3.477.162,10.060.000,4.000.000,available,Renta anual por propiedad/unidad según dimensión disponible.,,monthly_flow_semantic_split.csv,number
Renta por propiedad,Renta por propiedad — Lugar: Tigre 32,IS.RENT.BY_PROPERTY,Lugar,Tigre 32,ARS,1.056.984,1.601.571,4.105.421,11.243.850,10.990.000,available,Renta anual por propiedad/unidad según dimensión disponible.,,monthly_flow_semantic_split.csv,number


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/income_rent_by_property.csv')

## 4. Drilldown: OPEX por categoría

Este bloque debe ser legible para contadores.  
Etiquetas como `semantic_subbucket: services` deben traducirse a `Servicios` en el output humano.


In [5]:
opex_by_category = build_statement_table(
    metrics,
    [spec("Income", "OPEX por categoría", "OPEX por categoría", "IS.OPEX.BY_CATEGORY", 100,
          append_dimension=True,
          professional_comment="Impuestos, servicios, mantenimiento, legal y otros OPEX reales.")],
    years,
    include_debug_cols=True,
    drop_all_empty_years=True,
)
display_statement(opex_by_category, "OPEX por categoría", "Detalle dimensionado de costos de propiedad.", debug=True)
export_table(opex_by_category, repo_root, "income_opex_by_category.csv")


## OPEX por categoría

Detalle dimensionado de costos de propiedad.

section,line,metric_id,dimension_name,dimension_value,Currency,2022,2023,2024,2025,2026,value_status,professional_comment,caveat,source_table,format_hint
OPEX por categoría,OPEX por categoría — semantic_subbucket: Impuestos,IS.OPEX.BY_CATEGORY,semantic_subbucket,taxes,ARS,227.940,1.541.706,4.183.410,6.876.980,5.414.569,available,"Impuestos, servicios, mantenimiento, legal y otros OPEX reales.",,monthly_flow_semantic_split.csv,number
OPEX por categoría,OPEX por categoría — semantic_subbucket: Legal,IS.OPEX.BY_CATEGORY,semantic_subbucket,legal,ARS,37.700,277.093,1.044.210,147.000,s/d,available,"Impuestos, servicios, mantenimiento, legal y otros OPEX reales.",,monthly_flow_semantic_split.csv,number
OPEX por categoría,OPEX por categoría — semantic_subbucket: Mantenimiento,IS.OPEX.BY_CATEGORY,semantic_subbucket,maintenance,ARS,215.900,57.500,592.174,1.131.802,s/d,available,"Impuestos, servicios, mantenimiento, legal y otros OPEX reales.",,monthly_flow_semantic_split.csv,number
OPEX por categoría,OPEX por categoría — semantic_subbucket: Servicios,IS.OPEX.BY_CATEGORY,semantic_subbucket,services,ARS,105.726,516.242,2.246.508,3.799.578,1.324.903,available,"Impuestos, servicios, mantenimiento, legal y otros OPEX reales.",,monthly_flow_semantic_split.csv,number


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/income_opex_by_category.csv')

## 5. Comentarios profesionales

Estos comentarios son parte del output: ayudan a que la tabla no sea solo numérica.


In [6]:
income_comments = pd.DataFrame([
    {"tema": "Separación contable", "comentario": "Este reporte excluye funding, préstamos, repagos, dividendos y gasto personal. Su objetivo es aislar la operación patrimonial."},
    {"tema": "Margen", "comentario": "El margen operativo y OPEX/renta deben agregarse porque explican si el crecimiento nominal se traduce en eficiencia."},
    {"tema": "2024", "comentario": "Si 2024 aparece con OPEX alto, revisar impuestos, servicios, legal y unknowns. Puede ser un año de presión operativa o de clasificación pendiente."},
    {"tema": "USD operativo", "comentario": "Las líneas USD chicas en operating deben auditarse: pueden ser alquileres en USD, diferencias de cambio, o movimientos mal clasificados."},
    {"tema": "2026", "comentario": "Si 2026 es YTD, marcarlo explícitamente para no compararlo como año cerrado."},
    {"tema": "Moneda constante", "comentario": "Para evaluación económica real, considerar una versión ARS ajustada por inflación. Esta notebook mantiene nominal contractual."},
])
display(income_comments)
export_table(income_comments, repo_root, "income_professional_comments.csv")


,tema,comentario
0,Separación contable,"Este reporte excluye funding, préstamos, repag..."
1,Margen,El margen operativo y OPEX/renta deben agregar...
2,2024,"Si 2024 aparece con OPEX alto, revisar impuest..."
3,USD operativo,Las líneas USD chicas en operating deben audit...
4,2026,"Si 2026 es YTD, marcarlo explícitamente para n..."
5,Moneda constante,"Para evaluación económica real, considerar una..."


PosixPath('/home/matias/repos/accounting-backend/out/professional_pack/latest/tables/income_professional_comments.csv')

## 6. Export report

In [7]:
md_path = write_markdown_report(
    repo_root,
    "03_income_rent_and_operations.md",
    "03 — Income, rent and operations",
    [
        ("Professional scope", "Reporte operativo puro: renta, OPEX y resultado. No incluye funding, deuda, repagos, dividendos ni gasto personal."),
        ("Operating statement", operating_table),
        ("Rent by property", rent_by_property),
        ("OPEX by category", opex_by_category),
        ("QA", operating_qa),
        ("Professional comments", income_comments),
    ],
)
html_path = write_html_report(
    repo_root,
    "03_income_rent_and_operations.html",
    "03 — Income, rent and operations",
    [
        ("Professional scope", "Reporte operativo puro: renta, OPEX y resultado. No incluye funding, deuda, repagos, dividendos ni gasto personal."),
        ("Operating statement", operating_table),
        ("Rent by property", rent_by_property),
        ("OPEX by category", opex_by_category),
        ("QA", operating_qa),
        ("Professional comments", income_comments),
    ],
)
print("markdown:", md_path.relative_to(repo_root))
print("html:", html_path.relative_to(repo_root))


markdown: out/professional_pack/latest/markdown/03_income_rent_and_operations.md
html: out/professional_pack/latest/html/03_income_rent_and_operations.html
